In [1]:
import joblib
import numpy as np
import pandas as pd

In [3]:
# Cargar el modelo y los scalers
loaded_model = joblib.load('model.pkl')
loaded_scaler_X = joblib.load('scaler_X.pkl')
loaded_scaler_y = joblib.load('scaler_y.pkl')

In [4]:
# Las columnas de entrada en el orden correcto (sin 'charges')
feature_cols = ['sex', 'smoker', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest', 'age', 'bmi', 'children']

def predict_price(
    sex: float,
    smoker: float,
    region_northeast: float,
    region_northwest: float,
    region_southeast: float,
    region_southwest: float,
    age: float,
    bmi: float,
    children: float
) -> float:
    """
    Predice el costo del seguro basado en las características proporcionadas.

    Args:
        sex (float): 0.0 para femenino, 1.0 para masculino.
        smoker (float): 0.0 para no fumador, 1.0 para fumador.
        region_northeast (float): 1.0 si es noreste, 0.0 en otro caso.
        region_northwest (float): 1.0 si es noroeste, 0.0 en otro caso.
        region_southeast (float): 1.0 si es sureste, 0.0 en otro caso.
        region_southwest (float): 1.0 si es suroeste, 0.0 en otro caso.
        age (float): Edad del asegurado.
        bmi (float): Índice de Masa Corporal del asegurado.
        children (float): Número de hijos a cargo del asegurado.

    Returns:
        float: El costo predicho del seguro.
    """
    # Crear un array con los valores de entrada en el orden correcto
    input_data = np.array([
        sex,
        smoker,
        region_northeast,
        region_northwest,
        region_southeast,
        region_southwest,
        age,
        bmi,
        children
    ]).reshape(1, -1)

    # Escalar las características de entrada usando el scaler_X cargado
    input_scaled = loaded_scaler_X.transform(input_data)

    # Realizar la predicción con el modelo cargado
    predicted_charge_scaled = loaded_model.predict(input_scaled)

    # Invertir la escala de la predicción usando el scaler_y cargado
    predicted_charge = loaded_scaler_y.inverse_transform(predicted_charge_scaled.reshape(-1, 1))

    return predicted_charge[0][0]

# Ejemplo de uso de la función:
# Asumiendo un hombre, no fumador, en el suroeste, de 30 años, con BMI de 25 y 1 hijo.
# sex=1.0 (male), smoker=0.0 (no), region_northeast=0.0, region_northwest=0.0,
# region_southeast=0.0, region_southwest=1.0, age=30.0, bmi=25.0, children=1.0

sample_prediction = predict_price(1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 30.0, 25.0, 1.0)
print(f"El costo de seguro predicho para el ejemplo es: ${sample_prediction:.2f}")

# Otro ejemplo: mujer, fumadora, noreste, 45 años, BMI 30, 2 hijos
sample_prediction_2 = predict_price(0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 45.0, 30.0, 2.0)
print(f"El costo de seguro predicho para el segundo ejemplo es: ${sample_prediction_2:.2f}")

El costo de seguro predicho para el ejemplo es: $5079.86
El costo de seguro predicho para el segundo ejemplo es: $25019.64
